        # 🗂️ L09　元組、字典、集合
        **Python 冒險之旅 2026**　｜　Day 4（09/03 四）🏔️ 函式之島　｜　關卡　｜　🏅 100 XP

        📖 對應教科書：第 8 章 8.1–8.3


        ### 🎯 這一關你會學到
        - 分辨 tuple / dict / set 的使用時機
- 字典的新增、查詢、修改、刪除與走訪
- 集合運算：聯集、交集、差集

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/python-quest-2026/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  Python 冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins

_LEVEL = "L09"
_SALT = "python-quest-2026-datama"
_TASKS = ["9-1", "9-2", "9-3", "9-4", "9-5", "9-6"]
_XP_EACH = 16
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_pyquest_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

def 行列表(out):
    return [ln.rstrip() for ln in str(out).splitlines() if ln.strip()]

class _NeedMoreInput(Exception):
    pass

_HIST = builtins.__dict__.setdefault("_pyquest_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_pyquest_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_pyquest_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

def _find_cell(tid):
    marker = "# 🎯 任務 " + tid
    for cell in reversed(_history()):
        if marker in cell:
            lines = [ln for ln in cell.splitlines()
                     if not re.match(r"\s*(檢查|通關密語)\s*\(", ln)]
            return "\n".join(lines)
    return None

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                _plt.show = _orig_show
        return buf.getvalue(), ns
    run.src = src
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _progress():
    done = sum(1 for t in _TASKS if _PASSED.get(t))
    bar = "■" * done + "□" * (len(_TASKS) - done)
    return f"[{bar}] {done}/{len(_TASKS)}"

def 檢查(tid):
    tid = str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    src = _find_cell(tid)
    if src is None:
        print(f"❌ 找不到「# 🎯 任務 {tid}」的程式格。請先執行那一格（並保留第一行的標記），再執行這裡。")
        return
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        result = (False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。")
    except Exception as e:
        tb = traceback.format_exc().strip().splitlines()[-1]
        result = (False, f"程式執行時發生錯誤 → {tb}")
    ok, extra = (result, "") if isinstance(result, bool) else result
    if ok:
        first = not _PASSED.get(tid)
        _PASSED[tid] = True
        print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")
        if all(_PASSED.get(t) for t in _TASKS):
            print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
    else:
        print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
        if extra: print("   💬 " + str(extra))
        if _HINTS.get(tid): print("   💡 提示：" + _HINTS[tid])
        print("   👉 修改程式後，先重新執行任務那一格，再執行這一格。")

def 通關密語():
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_SALT}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：PYQ-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_9_1(run):
    out, ns = run()
    if (ns.get("r"), ns.get("g"), ns.get("b")) != (255, 128, 0): return (False, "r, g, b 應該是 255, 128, 0。")
    return (ns.get("rgba") == (255, 128, 0, 0.5), "rgba 應該是 (255, 128, 0, 0.5)，注意 (0.5,) 的逗號。")
任務定義("9-1", _check_9_1, 提示="r, g, b = rgb；rgba = rgb + (0.5,)。")

def _check_9_2(run):
    out, ns = run()
    return (ns.get("price") == {'汽水': 30, '冰棒': 20}, f"price 應該是 {{'汽水': 30, '冰棒': 20}}，現在是 {ns.get('price')}。")
任務定義("9-2", _check_9_2, 提示="price['冰棒'] = 20；price['汽水'] = 30；del price['公主麵']。")

def _check_9_3(run):
    out, ns = run()
    f = ns.get("freq", {})
    if f.get("日") != 5 or f.get("天") != 2 or f.get("是") != 1: return (False, "日 應該 5 次、天 2 次、是 1 次。")
    if " " in f: return (False, "不要把空白算進去。")
    return (出現(out, "最多的是日5次"), "最多的字是 日 5 次。")
任務定義("9-3", _check_9_3, 提示="freq[ch] = freq.get(ch, 0) + 1。")

def _check_9_4(run):
    out, ns = run("A001")
    if not 出現(out, "品名：汽水售價：25元"): return (False, "A001 應該印出 汽水 25元。")
    out, ns = run("X999")
    return (出現(out, "X999不存在"), "X999 應該印出 不存在。")
任務定義("9-4", _check_9_4, 提示="條件 num in datas。")

def _check_9_5(run):
    out, ns = run()
    if ns.get("dup") != {'王一', '張三', '趙六'}: return (False, "dup 應該是 {'王一', '張三', '趙六'}（交集 &）。")
    if ns.get("total") != 12: return (False, "total 應該是 12（聯集 | 的長度）。")
    return (ns.get("only1") == {'林二', '李四', '陳五'}, "only1 應該是 {'林二', '李四', '陳五'}（差集 -）。")
任務定義("9-5", _check_9_5, 提示="dup = s1 & s2；total = len(s1 | s2)；only1 = s1 - s2。")

def _check_9_6(run):
    out, ns = run()
    s = ns.get("lotto")
    if not isinstance(s, set): return (False, "lotto 要是集合（set）。")
    if len(s) != 7: return (False, "要有 7 個號碼。")
    return (all(isinstance(x, int) and 1 <= x <= 49 for x in s), "號碼要在 1～49 之間。")
任務定義("9-6", _check_9_6, 提示="while len(lotto) < 7: lotto.add(random.randint(1, 49))。")


## 🗂️ 9-1　元組（tuple）：不能改的串列
用 `()` 建立。**不能修改元素**（安全、速度快），適合放固定資料，例如座標、RGB 顏色、星期名稱。

In [ ]:
tuple1 = ('東', '南', '西')              # 課本 ex08/tuple_1.py
print(tuple1[0], len(tuple1), '東北' in tuple1)
East, South, West = tuple1              # 拆解（unpack）
print(South)
tuple2 = tuple1 + ('北',)               # 只有一個元素的元組要加逗號！
print(tuple2)
lst = list(tuple2); lst.append('東北'); tuple3 = tuple(lst)   # 要改就先轉成串列
print(tuple3, tuple3.index('北'), tuple3.count('東'))

## 9-2　字典（dict）：用「鍵」找「值」
用 `{鍵: 值}` 建立。鍵通常是字串或數字且**不可重複**；值可以是任何東西（包含串列、另一個字典）。
查詢速度超快，是資料處理最常用的結構之一（JSON 就是字典！）。

In [ ]:
dict1 = {'一月': '正月', '二月': '花月', '三月': '梅月'}     # 課本 ex08/dict_1.py
print(dict1['三月'])                   # 用鍵取值
dict1['一月'] = '端月'                  # 修改
dict1['四月'] = '桐月'                  # 新增
del dict1['二月']                      # 刪除
print(dict1, len(dict1))
print('一月' in dict1, '1月' in dict1)  # 鍵存在嗎？
print(dict1.get('五月'), dict1.get('五月', '查無'))   # get：找不到不會報錯
for k in dict1:                        # 走訪鍵
    print(k, dict1[k])
for k, v in dict1.items():             # 同時拿鍵與值
    print(f'{k} → {v}')
print(list(dict1.keys()), list(dict1.values()))

## 9-3　集合（set）：不重複、無順序
用 `{}` 或 `set()` 建立（空集合只能用 `set()`）。自動去除重複，支援聯集、交集、差集。

In [ ]:
g1 = ['林二', '王一', '張三', '趙六', '王一', '李四', '張三', '陳五']     # 課本 ex08/group.py
g2 = ['鄭十', '趙六', '劉千', '廖八', '柯七', '張三', '王一', '呂九', '柯七', '蔡百']
s1, s2 = set(g1), set(g2)
print(len(g1), len(s1))                # 8 6：去除重複
print(s1 & s2)                         # 交集：兩邊都有
print(s1 | s2)                         # 聯集
print(s1 - s2)                         # 差集：只在 s1
print(s1 ^ s2)                         # 對稱差：只在其中一邊
s1.add('新人'); s1.discard('不存在')    # 新增／移除
print({1, 2} <= {1, 2, 3})             # 子集合

| | 串列 list | 元組 tuple | 字典 dict | 集合 set |
|---|---|---|---|---|
| 符號 | `[ ]` | `( )` | `{鍵: 值}` | `{ }` |
| 有順序 | ✅ | ✅ | ✅(3.7+) | ❌ |
| 可修改 | ✅ | ❌ | ✅ | ✅ |
| 可重複 | ✅ | ✅ | 鍵不可 | ❌ |
| 用途 | 一般序列 | 固定資料 | 對照表、紀錄 | 去重、集合運算 |

### 🎯 任務 9-1　元組拆解與組合

`rgb = (255, 128, 0)`。把它拆解成 `r, g, b` 三個變數；再建立 `rgba = rgb + (0.5,)`；印出 `r g b` 與 `rgba` 的長度。預期：`255 128 0` 和 `4`。

In [ ]:
# 🎯 任務 9-1　元組拆解與組合（請保留這一行）
rgb = (255, 128, 0)
r, g, b = ???
rgba = ???
print(r, g, b)
print(len(rgba))

In [ ]:
檢查("9-1")   # ◀ 執行這一格，看看任務 9-1 有沒有過關

### 🎯 任務 9-2　字典 CRUD

從 `price = {'汽水': 25, '公主麵': 10}` 開始：新增 `冰棒: 20`；把 `汽水` 改成 30；刪除 `公主麵`；最後印出字典與 `'冰棒' in price`。

In [ ]:
# 🎯 任務 9-2　字典 CRUD（請保留這一行）
price = {'汽水': 25, '公主麵': 10}
# 新增、修改、刪除
print(price)
print('冰棒' in price)

In [ ]:
檢查("9-2")   # ◀ 執行這一格，看看任務 9-2 有沒有過關

### 🎯 任務 9-3　字頻統計

統計句子 `text` 中每個字（不含空白）出現的次數，存到字典 `freq`，並印出出現最多次的字與次數。提示：`freq[ch] = freq.get(ch, 0) + 1`。

In [ ]:
# 🎯 任務 9-3　字頻統計（請保留這一行）
text = "日日是好日 天天向上 日日進步"
freq = {}
for ch in text:
    if ch == ' ':
        continue
    ???
best = max(freq, key=freq.get)
print(freq)
print("最多的是", best, freq[best], "次")

In [ ]:
檢查("9-3")   # ◀ 執行這一格，看看任務 9-3 有沒有過關

### 🎯 任務 9-4　貨號查詢（輸入版）

`datas` 是貨號對應 [品名, 售價] 的字典。讀取貨號：存在就印 `貨號：A001 品名：汽水 售價：25元`；不存在印 `貨號：X999 不存在`。

In [ ]:
# 🎯 任務 9-4　貨號查詢（輸入版）（請保留這一行）
datas = {'A001': ['汽水', 25], 'A005': ['公主麵', 10], 'A006': ['口香糖', 8], 'A003': ['冰棒', 20]}
num = input('請輸入貨號：')
if ???:
    d = datas[num]
    print(f"貨號：{num} 品名：{d[0]} 售價：{d[1]}元")
else:
    print(f"貨號：{num} 不存在")

In [ ]:
檢查("9-4")   # ◀ 執行這一格，看看任務 9-4 有沒有過關

### 🎯 任務 9-5　社團名單分析

兩個社團名單 `g1`、`g2`（有人重複報名）。用集合算出：`dup`（兩社都參加的人）、`total`（合併後總人數）、`only1`（只在 g1 的人）。印出三者。

In [ ]:
# 🎯 任務 9-5　社團名單分析（請保留這一行）
g1 = ['林二', '王一', '張三', '趙六', '王一', '李四', '張三', '陳五']
g2 = ['鄭十', '趙六', '劉千', '廖八', '柯七', '張三', '王一', '呂九', '柯七', '蔡百']
s1, s2 = set(g1), set(g2)
dup = ???
total = ???
only1 = ???
print("重複參加：", dup)
print("合併人數：", total)
print("只在 g1：", only1)

In [ ]:
檢查("9-5")   # ◀ 執行這一格，看看任務 9-5 有沒有過關

### 🎯 任務 9-6　大樂透產生器

用 `random` 與 **集合** 產生 7 個 1～49 之間**不重複**的號碼，存到集合 `lotto`，最後排序印出。

In [ ]:
# 🎯 任務 9-6　大樂透產生器（請保留這一行）
import random
lotto = set()
while len(lotto) < ???:
    lotto.add(???)
print(sorted(lotto))

In [ ]:
檢查("9-6")   # ◀ 執行這一格，看看任務 9-6 有沒有過關

---
## 🔑 通關密語

全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：⚔️ B4 Boss 戰：小小點餐系統** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/python-quest-2026/blob/main/notebooks/B4_boss_ordering_system.ipynb)

回到入口網頁：https://johnnychao.github.io/python-quest-2026/